## Feature Scoping to determine which feature are important for the model

In [2]:
import json 
import numpy as np 
import pandas as pd 
from pathlib import Path 

ROOT = Path().cwd().parent

In [3]:
CLIENTS = ROOT / "data" / "processed" / "client"
labels = json.loads((ROOT / "data" / "processed" / "station_labels.json").read_text())

frames = {labels[p.stem]["label"]: pd.read_csv(p)
          for p in sorted(CLIENTS.glob("*.csv"))}

frames = dict(sorted(frames.items(), key=lambda kv: int(kv[0][1:])))   # S1..S7

print({k: v.shape for k, v in frames.items()})
print(list(frames["S1"].columns))


{'S1': (2470, 26), 'S2': (1627, 26), 'S3': (1789, 26), 'S4': (1769, 26), 'S5': (1770, 26), 'S6': (1759, 26), 'S7': (6799, 26)}
['station_hash_id', 'measured_ts', 'total_produced_energy', 'source', 'ac_power', 'dc_power', 'efficiency', 'device_temperature', 'perc_state_on', 'perc_state_off', 'perc_state_error', 'temperature_2m', 'shortwave_radiation', 'direct_radiation', 'diffuse_radiation', 'direct_normal_irradiance', 'global_tilted_irradiance', 'terrestrial_radiation', 'terrestrial_radiation_instant', 'global_tilted_irradiance_instant', 'direct_normal_irradiance_instant', 'diffuse_radiation_instant', 'direct_radiation_instant', 'shortwave_radiation_instant', 'tilt', 'azimuth']


In [4]:
audit = pd.DataFrame({sid: df.nunique() for sid, df in frames.items()}).T
audit

,station_hash_id,measured_ts,total_produced_energy,source,ac_power,dc_power,efficiency,device_temperature,perc_state_on,perc_state_off,...,global_tilted_irradiance,terrestrial_radiation,terrestrial_radiation_instant,global_tilted_irradiance_instant,direct_normal_irradiance_instant,diffuse_radiation_instant,direct_radiation_instant,shortwave_radiation_instant,tilt,azimuth
S1,1,2470,995,1,1458,1481,1259,1383,13,13,...,1368,1471,1224,101,12,492,38,2415,1,1
S2,1,1627,1039,1,1475,1490,1224,1398,26,26,...,1368,1471,1048,101,12,452,34,1605,1,1
S3,1,1789,1452,1,1649,1729,1579,1706,178,172,...,1508,1627,1116,101,12,453,35,1761,1,1
S4,1,1769,1192,1,1583,1575,1601,1722,204,204,...,1529,1620,637,101,12,489,25,1745,1,1
S5,1,1770,1444,1,1603,1684,1600,1727,213,213,...,1529,1620,637,101,12,490,25,1744,1,1
S6,1,1759,1328,1,1552,1671,1563,1727,207,207,...,1517,1607,637,101,12,490,25,1733,1,1
S7,1,6799,2952,1,4171,5702,5722,5877,1,1,...,3845,4151,2227,101,12,624,44,6548,1,1


## Cell 2 - Split "zero information" from "client identifier"

In [5]:
pooled = pd.concat(frames.values(), keys=frames.keys(), names=["station", "row"])

const_within = [c for c in audit.columns if (audit[c] <= 1).all()]

verdict = pd.DataFrame({
    "n_unique_pooled": pooled[const_within].nunique(),
})
verdict["verdict"] = np.where(
    verdict["n_unique_pooled"] <= 1,
    "globally constant — zero information, free drop",
    "CLIENT IDENTIFIER — varies only across clients, must drop",
)
verdict

,n_unique_pooled,verdict
station_hash_id,7,CLIENT IDENTIFIER — varies only across clients...
source,2,CLIENT IDENTIFIER — varies only across clients...
tilt,1,"globally constant — zero information, free drop"
azimuth,1,"globally constant — zero information, free drop"


## Cell 3 - State the decision rule before you lLookupError

- H1: Is terrestrial_radiation is the radation emitted by the sun?
- H2: Is it the radiation emitted by the Earth Surface?

If data shows radiation = 0 w/m^2 at night -> behaves like a solar radiation  
If the radation stays between 300 to 400 w/m^2 in day and night then it's terrestrial radiation

In [6]:
day   = pooled[pooled["global_tilted_irradiance"] >  0]
night = pooled[pooled["global_tilted_irradiance"] == 0]

for name, sub in [("day", day), ("night", night)]:
    s = sub["terrestrial_radiation"]
    print(f"{name:6s} mean={s.mean():8.1f}  max={s.max():8.1f}  frac_zero={(s == 0).mean():.3f}")

day    mean=   597.2  max=  1254.1  frac_zero=0.000
night  mean=     0.1  max=    25.1  frac_zero=0.974


## Cell 4 - Verify the radiation columns means what their names claim

In [7]:
resid = day["shortwave_radiation"] - (day["direct_radiation"] + day["diffuse_radiation"])
print(resid.abs().describe().round(3))

count    13444.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
dtype: float64


## Feature Engineering 
### Cell 5 - Two Physics Derived Feature